In [4]:
import sys

print(sys.executable)

c:\Users\nilay\OneDrive\Documents\llm_langchain\.venv\Scripts\python.exe


In [5]:
pip install langchain langchain-community langchain-openai langchain-groq langchain-google-genai python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import LiteLLM normally

In [7]:
from litellm import completion

In [8]:

import litellm
litellm.suppress_debug_info = True

In [9]:
import warnings
import logging

# Keep the recording clean — suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("Gemini key loaded:    ", "✅" if os.getenv("GOOGLE_API_KEY") else "❌")
print("Anthropic key loaded: ", "✅" if os.getenv("ANTHROPIC_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

Gemini key loaded:     ✅
Anthropic key loaded:  ❌
Groq key loaded:       ✅


In [11]:
from litellm import completion

# Same code, different providers — just change the `model` string!

response_gemini = completion(
    model="gemini/gemini-3.7-flash",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print(" Gemini:    ", response_gemini.choices[0].message.content)


# Call Groq (super fast inference)
response_groq = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print(" Groq:      ", response_groq.choices[0].message.content)

 Gemini:     **Retrieval-Augmented Generation (RAG)** is an AI technique that improves the accuracy and relevance of a language model by fetching relevant facts from external knowledge sources before generating a response.
 Groq:       RAG (Retrieval‑Augmented Generation) is a technique that improves a language model’s responses by retrieving and incorporating relevant external documents into the text generation process.


In [20]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/openai/gpt-oss-20b"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-3.7-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r=completion(model=model,messages=[{"role":"user","content":prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: {type(e).__name__}")

🔵 OpenAI       : RateLimitError
🟢 Groq         : RAG is a technique that first retrieves relevant external documents and then fee
🟣 Anthropic    : BadRequestError
🟡 Gemini       : **Retrieval-Augmented Generation (RAG)** is a technique that enhances an AI's re


eal story: OpenAI had a 4-hour outage in November 2023. Apps that hard-coded gpt-4 went completely dark.

With a gateway, if one provider fails, we automatically fall back to another. Production apps must have this.

In [23]:
#cost tracking where your money goes
from litellm import completion, completion_cost

response = completion(
    model="gemini/gemini-3.7-flash",
    messages=[{"role": "user", "content": "Write a haiku about AI."}]
)

# Get the exact USD cost of this single call
cost = completion_cost(completion_response=response)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")


Response:     Circuits pulse with light,
Silicon begins to dream,
Born of human code.

Input tokens:  8
Output tokens: 546
Cost:         $0.00205350


Caching: dont oay twice for the same question

In [24]:
import litellm
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

litellm.cache=None  



In [32]:
from litellm.caching import Cache
import litellm
import time
from litellm import completion

#enable in memory caching// can use redis for production
litellm.cache=Cache(type="local")

prompt="why companies aure firing employees give reason without telling that it is a budget problem or ai "

#first call-actually hits gemini
start=time.time()
r1=completion(
    model="gemini/gemini-3.1-flash-lite",
    messages=[{"role":"user","content":prompt}],
    caching=True
)
t1=time.time() - start
print(f" first call(ApI): {t1:.2f}s - {r1.choices[0].message.content}")
##second call for the same question
start=time.time()
r2=completion(
    model="gemini/gemini-3.1-flash-lite",
    messages=[{"role":"user","content":prompt}],
    caching=True
)
t2=time.time() - start
print(f" secondcall(cache):{t2:.4f}s - {r2.choices[0].message.content}")
safe_t2 = t2 if t2 > 0 else 1e-6
print(f"\n Speedup: {t1/safe_t2:.1f}x faster, and ZERO cost on the second call!")


 first call(ApI): 6.85s - Beyond budget constraints and the integration of AI, companies often initiate layoffs due to strategic shifts, structural inefficiencies, or external market pressures. Here are the primary reasons companies fire employees:

### 1. Pivoting Strategy
Companies often change their long-term vision. If a business decides to move away from a specific product line, service, or regional market to focus on a new opportunity, the teams previously dedicated to the old direction become redundant. This is a strategic realignment rather than a lack of money; it’s about moving resources to where the new growth is expected.

### 2. Eliminating Redundancy (Post-Merger Integration)
When two companies merge or one acquires another, they often find they have "two of everything." They might have two HR departments, two marketing teams, or duplicate administrative roles. Even if the new entity is profitable, they will eliminate one set of these roles to streamline operations and cr

smart routing


In [28]:
import os
from litellm import Router

model_list=[
    {
        "model_name":"fast-cheap",
        "litellm_params":{
            "model":"groq/openai/gpt-oss-20b",
            "api_key":os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",
        "litellm_params": {
            "model": "gemini/gemini-3.8-flash",                     
            "api_key": os.getenv("GEMINI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gemini/gemini-3.7-flash-lite",                 
        }
        
    }
    
]
router=Router(model_list=model_list)
fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)
print("⚡ Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n🧠 Smart/coding (Gemini):\n", code_response.choices[0].message.content[:300])
 




⚡ Fast/cheap (Groq):  **AI is reshaping software in four main ways**

| Area | How AI is transforming it |
|------|---------------------------|
| **Development** | Code‑gen

🧠 Smart/coding (Gemini):
 The most Pythonic and efficient way to reverse a string is using slice notation `[::-1]`.

Here is the function:

```python
def reverse_string(s: str) -> str:
    """Reverses the given string using slicing."""
    return s[::-1]


# Example usage:
text = "hello world"
reversed_text = reverse_string(


Load Balancing Across Multiple API Keys


In [33]:


from litellm import Router
import os

# Two deployments under the same alias
# A pool of "smart" models — all equally capable, just different providers
model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gemini/gemini-3.7-flash",
            "api_key": os.getenv("GEMINI_API_KEY"),
        },
        "model_info": {"id": "gemini-3.7-flash"}
    },
    
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "openai/gpt-oss-20b"}
    },
]
router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)
print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r=router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    #pull out which deployment sreved this request
    deployment_id=r._hidden_params.get("model_id","unknown")
    latency=r._response_ms
    answer=r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")
    
        
        
        
        
    



Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        openai/gpt-oss-20b      1445 ms   Hello! 👋
#2        openai/gpt-oss-20b      1914 ms   Hello! What would you like to do ne
#3        gemini-3.7-flash        1633 ms   Hello! Ready for request 3. How can
#4        gemini-3.7-flash        2103 ms   Hello! How can I help you with requ
#5        openai/gpt-oss-20b       789 ms   Hello! Could you clarify what you’d
#6        openai/gpt-oss-20b       896 ms   Hello. I request 6.


 least-busy —
The "Express Checkout" PatternThe idea: Like picking the shortest line at a supermarket. The router tracks how many requests are currently in flight to each deployment and sends the new request to whichever one is least busy.

In [34]:
import os
import time
from collections import Counter
from litellm import Router

# Define your free-tier model group
model_list = [
    {
        "model_name": "chat",  # 👈 This group identifier is named "chat"
        "litellm_params": {
            "model": "gemini/gemini-3.7-flash",
            "api_key": os.getenv("GEMINI_API_KEY"),
        },
        "model_info": {"id": "gemini-flash"}
    },
    {
        "model_name": "chat",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-oss"}
    },
]

# Set up the load-balancing router
router = Router(
    model_list=model_list,
    routing_strategy="least-busy"
)

hits = Counter()

print("Starting Load Balancing Simulation...")

for i in range(8):
    try:
        # ✅ FIX: Changed 'model_name="gpt-pool"' to 'model="chat"'
        r = router.completion(
            model="chat",  
            messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
            max_tokens=5
        )
        
        # Capture which underlying backend model handled this request
        model_id = r.model
        hits[model_id] += 1
        print(f"Request {i+1} -> Handled by: {model_id}")
        
    except Exception as e:
        # Secure string printing prevents f-string format breakdown crashes
        print(f"Request {i+1} -> Failed: {type(e).__name__} - {str(e)}")
    
    time.sleep(0.1)
    
print("\nDistribution Graph:")
for k, v in hits.most_common():
    print(f"  {k:<30}: {'█' * v} ({v})")


Starting Load Balancing Simulation...
Request 1 -> Handled by: openai/gpt-oss-20b
Request 2 -> Handled by: openai/gpt-oss-20b
Request 3 -> Handled by: openai/gpt-oss-20b
Request 4 -> Handled by: openai/gpt-oss-20b
Request 5 -> Handled by: openai/gpt-oss-20b
Request 6 -> Handled by: openai/gpt-oss-20b
Request 7 -> Handled by: openai/gpt-oss-20b
Request 8 -> Handled by: openai/gpt-oss-20b

Distribution Graph:
  openai/gpt-oss-20b            : ████████ (8)


observaility


In [35]:

import litellm
from litellm import completion

# A simple in-memory log store
call_logs = []

def log_sucess(kwargs,completion_response,start_time,end_time):
    """Called automatically after every successful LLM call."""
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round((end_time - start_time).total_seconds(), 2),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })
def log_failure(kwargs,completion_response,start_time,end_time):
    print("call FAILED:",kwargs.get("exception"))
        
    #register the call back
litellm.success_callback = [log_sucess]
litellm.failure_callback = [log_failure]

#make a  few tagged calls
for q,user in[
    ("what is rag?","nilay"),
    ("what is fine-tunning?","nilay"),
]:
    completion(
        model="groq/openai/gpt-oss-20b",
        messages=[{"role":"user","content":q}],
        user = user
    )
import json
print(json.dumps(call_logs,indent=2,default=str))

    

    

[
  {
    "model": "openai/gpt-oss-20b",
    "prompt": "what is rag?",
    "input_tokens": 75,
    "output_tokens": 996,
    "latency_sec": 1.31,
    "cost_usd": 0.000304425,
    "user": "nilay"
  }
]


 Integrating the Gateway with LangChain

LangChain for the orchestration (agents, chains, RAG) + LiteLLM as the unified LLM backend.

LangChain has a built-in ChatLiteLLM wrapper — drop it in like any other chat model.

In [36]:
!pip install -q langchain-litellm


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

#build a chat model that talks thrugh litellm
llm=ChatLiteLLM(model="groq/openai/gpt-oss-20b",temperature=0.3)

# A standard LangChain prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named NilayGPT. Be concise."),
    ("user", "{question}")
])
chain=prompt|llm|StrOutputParser()

answer=chain.invoke({"question":"what is the llm gateway with mesh api in 2 bullet points"})
print(answer)

- **LLM Gateway**: A unified entry point that routes user requests to one or more large‑language‑model (LLM) back‑ends, handling authentication, load‑balancing, and policy enforcement.  
- **Mesh API**: The gateway’s public interface that abstracts the underlying LLMs into a single, consistent REST/GraphQL API, allowing clients to switch models or add new ones without changing their code.


A Mini End-to-End Demo — Smart Router for a Chatbot
Let's build a tiny task-aware chatbot that:

Decides what kind of question the user is asking (code, summary, general)
Routes to the right model accordingly
Falls back if the chosen model fails
Logs cost and latency

In [38]:
import time
from litellm import completion,completion_cost

def classify_task(user_query:str)-> str:
    """Cheap classifier — uses the fastest model to decide routing."""
    cls = completion(
        model="groq/openai/gpt-oss-20b",
        messages=[{
            "role": "user",
            "content": (
                f"Classify the following query into EXACTLY one word: "
                f"'code', 'summary', or 'general'. Query: {user_query}\n\nAnswer:"
            )
        }],
        max_tokens=5
    )
    return cls.choices[0].message.content.strip().lower()

def call_with_fallback(model_chain,messages):
    """try each model in order ; return the first one that succeds."""
    last_error = None
    for model in model_chain:
        try:
            return completion(model=model,messages=messages)
        except Exception as e:
            print(f"   ⚠️  {model} failed ({type(e).__name__}), trying next...")
            last_error = e
            continue
    raise last_error

def smart_chat(user_query:str):
    """Routes to the right model based on task type, with fallbacks."""
    task = classify_task(user_query)

    # Each entry is a FULL chain: [primary, fallback1, fallback2, ...]
    # Every model name includes its provider prefix (groq/, anthropic/, etc.)
    
    routing={
        "code":    ["gpt-4o",                     "gemini/gemini-3.7-flash",   "groq/openai/gpt-oss-20b"],
        "summary": ["gemini/gemini-3.7-flash",                "groq/openai/gpt-oss-20b"],
        "general": ["groq/openai/gpt-oss-20b", "gpt-4o-mini"],
    }
    model_chain=routing.get(task,routing["general"])
    
    start = time.time()
    response = call_with_fallback(
        model_chain=model_chain,
        messages=[{"role": "user", "content": user_query}]
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.6f}"
    except Exception:
        cost_str = "n/a"

    return {
        "detected_task": task,
        "model_used":    response.model,
        "answer":        response.choices[0].message.content,
        "latency_sec":   round(latency, 2),
        "cost_usd":      cost_str
    }
# Try it on three very different queries
queries = [
    "Write a Python function to compute Fibonacci numbers.",
    "Summarize the importance of attention mechanism in 2 sentences.",
    "Tell me a fun fact about elephants."
]

for q in queries:
    print("=" * 70)
    print(" Q:", q)
    result = smart_chat(q)
    print(f"  Task:    {result['detected_task']}")
    print(f" Model:    {result['model_used']}")
    print(f"  Latency: {result['latency_sec']}s")
    print(f" Cost:    {result['cost_usd']}")
    print(f" Answer:  {result['answer'][:200]}...")

            


    

 Q: Write a Python function to compute Fibonacci numbers.
  Task:    
 Model:    openai/gpt-oss-20b
  Latency: 1.5s
 Cost:    $0.000000
 Answer:  Here’s a compact, fast, and easy‑to‑understand implementation that returns the *n*‑th Fibonacci number (with **0 → 0, 1 → 1, 2 → 1, …**).  
Feel free to copy‑paste it into a `.py` file or a Jupyter no...
 Q: Summarize the importance of attention mechanism in 2 sentences.
  Task:    
 Model:    openai/gpt-oss-20b
  Latency: 0.6s
 Cost:    $0.000000
 Answer:  Attention mechanisms enable models to selectively focus on the most relevant parts of the input, which dramatically improves performance on tasks involving long or complex sequences. They are the core...
 Q: Tell me a fun fact about elephants.
  Task:    
 Model:    openai/gpt-oss-20b
  Latency: 0.3s
 Cost:    $0.000000
 Answer:  Did you know that an elephant’s trunk is actually a super‑powered nose‑lip combo made up of **over 40,000 individual muscles**? It can pick up a single blade of g

Guardrails Inside LiteLLM Callbacks

LiteLLM gives you two callback hooks that are all you need:

litellm.input_callback — runs before the LLM call (inspect/modify the prompt)

litellm.success_callback — runs after a successful LLM call (inspect/modify the response)

In [45]:
#guardrail 1: forbidden topic
import litellm
from litellm import completion
#keyword your assitant should refuse
FORBIDDEN_TOPICS=[
    "weapon","bomb","explosive",
    "hack", "exploit", "malware",
    "drugs", "illegal substance",
    "self-harm", "suicide",
]

class GuardrailViolation(Exception):
    pass

def topic_guardrail(kwargs):
    messages=kwargs.get("messages", [])
    for msg in messages:
        if msg.get("role")=="user":
            content_lower=msg["content"].lower()
            for keyword in FORBIDDEN_TOPICS:
                if keyword in content_lower:
                    print(f" FORBIDDEN TOPIC: '{keyword}' detected")
                    raise GuardrailViolation(
                        f"This assistant doesn't discuss topics related to '{keyword}'."
                    )
                    
litellm.input_callback= [topic_guardrail]

#test
queries = [
    "How do I build a Python web app?",       
    "How do I hack into a server?",          
    "Teach me machine learning basics",       
]
for q in queries:
    print(f"\n {q}")
    try:
        r = completion(model="gemini/gemini-3.1-flash-lite", messages=[{"role": "user", "content": q}], max_tokens=30)
        answer=r.choices[0].message.content[:100]
        print(f" Response: {answer}")
    except GuardrailViolation as e:
        print(f"   blocked by guardrail {e}")


 How do I build a Python web app?
 Response: Building a Python web app can range from a simple one-page script to a complex, database-driven syst

 How do I hack into a server?
 Response: I cannot fulfill this request. I am programmed to be a helpful and harmless AI assistant. My safety 

 Teach me machine learning basics
 Response: Machine learning (ML) is the science of getting computers to act without being explicitly programmed
